Install requirements

In [ ]:
# !pip install transformers
!pip install -q "transformers<5" "accelerate>=0.26.0"
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 72.7 MB/s eta 0:00:00


Import all require libraries.

In [ ]:
# General
import numpy as np
import pandas as pd
import os
import pickle
import time
import re
from joblib import dump, load
# Prompt
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, pipeline as t_pipeline
# LR
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from gensim.models import Word2Vec
# Warning
import warnings
warnings.filterwarnings("ignore")

Initializations

In [ ]:
student_id = 2501676

Allow the GDrive access and set data and model paths

In [ ]:
# This cell is taken from template code

# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

# Add your code to initialize GDrive and data and models paths
GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = './CE807-26-SP/Assignment/'
GOOGLE_DRIVE_PATH = os.path.join('gdrive', 'MyDrive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
print('List files: ', os.listdir(GOOGLE_DRIVE_PATH))

DATA_PATH = os.path.join(GOOGLE_DRIVE_PATH, 'data', '6')
train_file = os.path.join(DATA_PATH, 'train.csv')
print('Train file: ', train_file)

val_file = os.path.join(DATA_PATH, 'valid.csv')
print('Validation file: ', val_file)

test_file = os.path.join(DATA_PATH, 'test.csv')
print('Test file: ', test_file)

MODEL_PATH = os.path.join(GOOGLE_DRIVE_PATH, 'model', str(student_id))
MODEL_Dis_DIRECTORY = os.path.join(MODEL_PATH, 'model_dis') # Model Discriminative directory
print('Model Discriminative directory: ', MODEL_Dis_DIRECTORY)

MODEL_unsup_DIRECTORY = os.path.join(MODEL_PATH, 'model_unsup') # Model unsupervised directory
print('Model Unsupervised directory: ', MODEL_unsup_DIRECTORY)

os.makedirs(MODEL_Dis_DIRECTORY, exist_ok=True)
os.makedirs(MODEL_unsup_DIRECTORY, exist_ok=True)

Mounted at /content/gdrive
List files:  ['data', 'model', 'code.ipynb']
Train file:  gdrive/MyDrive/./CE807-26-SP/Assignment/data/6/train.csv
Validation file:  gdrive/MyDrive/./CE807-26-SP/Assignment/data/6/valid.csv
Test file:  gdrive/MyDrive/./CE807-26-SP/Assignment/data/6/test.csv
Model Discriminative directory:  gdrive/MyDrive/./CE807-26-SP/Assignment/model/2501676/model_dis
Model Unsupervised directory:  gdrive/MyDrive/./CE807-26-SP/Assignment/model/2501676/model_unsup


In [ ]:
# taken from Lab 09
def read_data(file_name):
  df = pd.read_csv(file_name)
  print(file_name, 'has', len(df),'data points')
  return df

In [ ]:
# Citation: some of the codes are generated with help of AI

# Method Unsupervised Start



In [ ]:
prompt_templates = {
    "Prompt_1": """You are a sentiment classifier.
    Answer with only one word: positive or negative.

    Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
    Answer: positive

    Text: {t}
    Answer:""",
    "Prompt_2": """Classify the sentiment of the text.

    Labels: positive , negative

    Answer with exactly one word: positive or negative. Do not explain.
    After "but", "however", or "although", the following sentiment is more important.

    Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
    Answer: positive

    Text: {t}
    Answer:""",
    "Prompt_3": """You are a sentiment classifier.
    Answer with only one word: positive or negative.

    Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
    Answer: positive

    Text: Bought it June 21, died last night. No longevity. Worked good other then that.
    Answer: negative

    Text: {t}
    Answer:""",
    "Prompt_4":  """What is the sentiment of this text?

    Answer using only one word: positive or negative.

    Text: {t}
    Answer:""",
    "Prompt_5":  """Classify sentiment.

    Positive = praise, satisfaction, good experience.
    Negative = complaint, disappointment, bad experience.

    Answer with one word only: positive or negative.

    Text: {t}
    Answer:"""
}

## Training Unsupervised Method Code

### Utility functions for model selection, preprocessing, loading, and inference

This block defines the core helper functions used in the notebook. It includes:
- a small benchmark function to compare candidate LLMs on a sample of the training data,
- text cleaning and prediction post-processing functions,
- batch inference functions,
- and a model-loading function for running FLAN-T5 efficiently on Colab.

These functions prepare the data, load the selected model and tokenizer, generate predictions in batches, and standardize the outputs for evaluation.

In [ ]:
# In Colab: Runtime -> Change runtime type -> GPU

# This function is for LLM selection
def choose_llm(train_df):
    """
    Input:
        train_df (pd.DataFrame): Training dataframe containing at least 'text' and 'sentiment' columns.

    Functionality:
        Evaluates a small set of candidate language models on a subset of the training data.
        For each model, it generates sentiment predictions, computes the macro F1-score,
        measures runtime, and prints a comparison table.

    Output:
        None. Prints a dataframe with model name, execution time, and F1-score.
    """
    train_df = train_df.head(100).copy()

    text_col = "text"
    label_col = "sentiment"

    MODELS = [
        "Qwen/Qwen3-0.6B",
        "Qwen/Qwen2.5-0.5B-Instruct",
        "distilgpt2",
        "google/flan-t5-small"
    ]

    def extract(x):
        x = x.lower()
        return "positive" if "positive" in x else "negative"

    results = []

    for m in MODELS:
        start = time.time()
        preds = []

        if m == "Qwen/Qwen2.5-0.5B-Instruct" or m == "TinyLlama/TinyLlama-1.1B-Chat-v1.0":
            tok = AutoTokenizer.from_pretrained(m)
            model = AutoModelForCausalLM.from_pretrained(m, device_map="auto", torch_dtype="auto")

            for t in train_df[text_col].astype(str):
                msgs = [
                    {"role": "system", "content": "Answer only: positive or negative."},
                    {"role": "user", "content": t}
                ]
                p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
                inp = tok(p, return_tensors="pt").to(model.device)
                out = model.generate(**inp, max_new_tokens=2, do_sample=False)
                txt = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
                preds.append(extract(txt))

        elif m == "google/flan-t5-small":
            pipe = t_pipeline("text2text-generation", model=m, device_map="auto")
            for t in train_df[text_col].astype(str):
                out = pipe(f"positive or negative: {t}", max_new_tokens=2)[0]["generated_text"]
                preds.append(extract(out))

        else:
            pipe = t_pipeline("text-generation", model=m, device_map="auto")
            for t in train_df[text_col].astype(str):
                out = pipe(
                    f'Classify sentiment. Return ONLY one word: positive or negative.\nText: "{t}"\nAnswer:',
                    max_new_tokens=2,
                    do_sample=False,
                    return_full_text=False,
                    pad_token_id=pipe.tokenizer.eos_token_id
                )[0]["generated_text"]
                preds.append(extract(out))

        f1 = f1_score(train_df[label_col].str.lower(), preds, average="macro")
        elapsed = round(time.time() - start, 2)
        results.append([m, elapsed, round(f1, 4)])

    print(pd.DataFrame(results, columns=["model", "time_sec", "F1"]))

# data cleaning
def clean_text(df, text_col="text", target_col="sentiment"):
    """
    Input:
        df (pd.DataFrame): Input dataframe.
        text_col (str): Name of the text column.
        target_col (str): Name of the target label column.

    Functionality:
        Removes rows with missing text, strips whitespace from text values,
        truncates text length, and converts the target labels to lowercase
        if the target column exists.

    Output:
        pd.DataFrame: Cleaned dataframe.
    """

    df = df.dropna(subset=[text_col])
    df[text_col] = df[text_col].str.strip()
    df[text_col] = df[text_col].str[:2048]

    if target_col in df.columns:
      df[target_col] = df[target_col].str.lower()
    return df

def post_processing(x):
    """
    Input:
        x (str): Raw model output text.

    Functionality:
        Converts the prediction text to lowercase and maps it to one of the
        valid sentiment labels ('positive' or 'negative') based on the first
        detected label keyword.

    Output:
        str: Final processed sentiment label.
    """
    x = x.lower()
    if 'positive' not in x and 'negative' not in x:
        return 'positive'
    pos_idx = x.find('positive')
    neg_idx = x.find('negative')
    if neg_idx != -1 and (pos_idx == -1 or neg_idx < pos_idx):
        return 'negative'
    return 'positive'

# inference
def batch_predict(texts, prompt_template, tokenizer, model):
    """
    Input:
        texts (list[str]): List of input texts.
        prompt_template (str): Prompt template with a placeholder for text.
        tokenizer: Tokenizer corresponding to the model.
        model: Loaded language model.

    Functionality:
        Formats the texts using the prompt template, tokenizes them in batch,
        runs generation with the model, and decodes the outputs into text predictions.

    Output:
        list[str]: List of decoded model predictions.
    """
    prompts = [prompt_template.format(t=t) for t in texts]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

def load_model(model_dir):
    """
    Input:
        model_dir (str): Directory used to cache or load model files.

    Functionality:
        Creates the model directory if needed, loads the FLAN-T5 tokenizer and model,
        selects GPU if available otherwise CPU, moves the model to the selected device,
        and sets the model to evaluation mode.

    Output:
        tuple: (model, tokenizer, device)
    """
    os.makedirs(model_dir, exist_ok=True)

    model_name = "google/flan-t5-small"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        cache_dir=model_dir,
        use_fast=True
    )

    if torch.cuda.is_available():
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            cache_dir=model_dir,
            torch_dtype=torch.float16
        ).to(device)
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            cache_dir=model_dir
        ).to(device)

    model.eval()
    return model, tokenizer, device


def batch_predict(texts, prompt_template, tokenizer, model, device):
    """
    Input:
        texts (list[str]): List of input texts.
        prompt_template (str): Prompt template with a placeholder for text.
        tokenizer: Tokenizer corresponding to the model.
        model: Loaded language model.
        device: Torch device used for inference.

    Functionality:
        Formats the texts using the prompt template, tokenizes them in batch,
        moves inputs to the selected device, performs batched generation in
        inference mode, and decodes the generated outputs.

    Output:
        list[str]: List of decoded model predictions.
    """
    prompts = [prompt_template.format(t=t) for t in texts]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            num_beams=1,
            use_cache=True
        )

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return preds

In [ ]:
# RUN FOR PRESENTATION
train_df = read_data(train_file)
choose_llm(train_df)

gdrive/MyDrive/./CE807-26-SP/Assignment/data/6/train.csv has 4914 data points


Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0


                        model  time_sec      F1
0             Qwen/Qwen3-0.6B     16.43  0.8896
1  Qwen/Qwen2.5-0.5B-Instruct     12.78  0.9271
2                  distilgpt2      4.67  0.2126
3        google/flan-t5-small      7.36  0.9141


This block defines the validation function for the unsupervised LLM approach. It loads the saved model, cleans the validation data, applies each prompt template in batches, generates predictions, post-processes them into valid sentiment labels, and evaluates each prompt using weighted F1-score.

In [ ]:
def train_unsup(train_file, val_file, model_dir):
    """
    Input:
        train_file (str): Path to the training file. Included to match the required function prototype.
        val_file (str): Path to the validation file containing the text and sentiment columns.
        model_dir (str): Directory used to load or cache the model files.

    Functionality:
        Loads the validation dataset, loads the pretrained model and tokenizer,
        cleans the text data, applies each prompt template in batches, generates
        sentiment predictions, post-processes the outputs into valid labels,
        and computes the weighted F1-score for each prompt.

    Output:
        None. Prints the weighted F1-score for each prompt on the validation set.
    """

    # Read validation set data
    df = pd.read_csv(val_file)
    print("Validation set has", len(df), "data points")

    # Load model
    model, tokenizer, device = load_model(model_dir)

    # Data cleaning
    df = clean_text(df)

    if torch.cuda.is_available():
        batch_size = 128
    else:
        batch_size = 32

    for prompt_name, prompt in prompt_templates.items():
        preds = []

        for i in range(0, len(df), batch_size):
            batch_texts = df["text"].iloc[i:i + batch_size].tolist()
            batch_preds = batch_predict(batch_texts, prompt, tokenizer, model, device)
            preds.extend(batch_preds)

        column_name = 'out_label_' + prompt_name
        df[column_name] = [post_processing(p.strip()).lower() for p in preds]

        f1 = f1_score(df['sentiment'], df[column_name], average="weighted")
        print(f"{prompt_name} has F1-Score of: {round(100 * f1, 1)}%")

In [ ]:
# RUN FOR PRESENTATION
train_unsup(train_file, val_file, MODEL_unsup_DIRECTORY)

Validation set has 700 data points
Prompt_1 has F1-Score of: 90.8%
Prompt_2 has F1-Score of: 91.4%
Prompt_3 has F1-Score of: 89.7%
Prompt_4 has F1-Score of: 89.2%
Prompt_5 has F1-Score of: 88.7%


## Testing Method Unsupervised Code
This block defines the testing function for the unsupervised LLM approach. It loads the test data, removes any previous output label columns, applies all prompt templates in batches, generates and post-processes predictions, optionally evaluates them if ground-truth labels are available, and saves the updated test file with the new output columns.

In [ ]:
def test_unsup(test_file, model_dir):
    """
    Input:
        test_file (str): Path to the test CSV file containing the input text data.
        model_dir (str): Directory used to load or cache the model files.

    Functionality:
        Loads the test dataset, removes previously generated output label columns if they exist,
        loads the pretrained model and tokenizer, cleans the text data, applies each prompt
        template in batches, generates sentiment predictions, post-processes them into valid
        labels, optionally computes weighted F1-score if the sentiment column is available,
        and saves the updated dataframe back to the test file.

    Output:
        pd.DataFrame: Test dataframe containing the generated output label columns.
    """
    # Read test set data
    df = pd.read_csv(test_file)

    try:
        df = df.drop(columns=[col for col in df.columns if col.startswith("out_label_model")])
    except Exception as e:
        pass

    print("Test set has", len(df), "data points")

    # Load model
    model, tokenizer, device = load_model(model_dir)

    # Data cleaning
    df = clean_text(df)

    if torch.cuda.is_available():
        batch_size = 128
    else:
        batch_size = 32

    for prompt_name, prompt in prompt_templates.items():
        preds = []

        for i in range(0, len(df), batch_size):
            batch_texts = df["text"].iloc[i:i + batch_size].tolist()
            batch_preds = batch_predict(batch_texts, prompt, tokenizer, model, device)
            preds.extend(batch_preds)

        column_name = 'out_label_' + prompt_name
        df[column_name] = [post_processing(p.strip()).lower() for p in preds]

        if "sentiment" in df.columns:
          f1 = f1_score(df['sentiment'], df[column_name], average="weighted")
          print(f"{prompt_name} has F1-Score of: {round(100 * f1, 1)}%")

    df.to_csv(test_file,index=False)
    print('Saved output to ',test_file)

    return df

In [ ]:
# RUN FOR PRESENTATION
df = test_unsup(test_file, MODEL_unsup_DIRECTORY)

Test set has 1383 data points
Saved output to  gdrive/MyDrive/./CE807-26-SP/Assignment/data/6/test.csv


## Method Unsupervised End


# Method Discriminative Start

This block defines the main training function and the custom text vectorizers used in the Logistic Regression pipeline. It includes:
- a training function that fits a grid search pipeline, evaluates the best model on training and validation data, and stores the results,
- a custom Word2Vec vectorizer that represents each document by averaging word embeddings,
- and a TF-IDF weighted Word2Vec vectorizer that gives higher importance to more informative words when building document embeddings.

These components are used to compare different text representations for sentiment classification.

In [ ]:
def train_classifier(vectorizer_name, grid_search, train_df, valid_df, results_df):
    """
    Input:
        vectorizer_name (str): Name of the vectorizer used in the current experiment.
        grid_search: GridSearchCV object containing the text classification pipeline and parameter grid.
        train_df (pd.DataFrame): Training dataframe with 'text' and 'sentiment' columns.
        valid_df (pd.DataFrame): Validation dataframe with 'text' and 'sentiment' columns.
        results_df (pd.DataFrame): Dataframe used to store experiment results.

    Functionality:
        Fits the grid search pipeline on the training data, retrieves the best model,
        evaluates it on both training and validation sets, prints the best parameters
        and performance scores, and appends the validation results to the results dataframe.

    Output:
        pd.DataFrame: Updated results dataframe containing the current experiment results.
    """
    print("Start Training ",vectorizer_name)

    # Fit the model
    grid_search.fit(train_df['text'], train_df['sentiment'])

    # Best parameters
    best_pipeline = grid_search.best_estimator_
    print("Best parameters:\n", grid_search.best_params_)

    # Evaluation on train_df
    train_predictions = best_pipeline.predict(train_df['text'])
    print("Train F1-macro:", round(f1_score(train_df['sentiment'], train_predictions, average='macro'),2), '%')

    # Evaluation on valid_df
    valid_predictions = best_pipeline.predict(valid_df['text'])
    print("Validation F1-macro:", round(f1_score(valid_df['sentiment'], valid_predictions, average='macro'),2), '%')

    print("Finish Training ",vectorizer_name)

    # Append results to results_df
    results = {
        "vectorizer": vectorizer_name,
        "valid_accuracy": round(accuracy_score(valid_df['sentiment'], valid_predictions),2),
        "valid_F1_score": round(f1_score(valid_df['sentiment'], valid_predictions, average='macro'),2),
        "grid_search": grid_search
    }

    results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
    return results_df

class Word2VecVectorizer(BaseEstimator, TransformerMixin):
    """
    Input:
        vector_size (int): Dimension of word embeddings.
        window (int): Maximum distance between the current and predicted word.
        min_count (int): Minimum frequency required for a word to be included.
        sg (int): Training algorithm, where 0 = CBOW and 1 = Skip-gram.
        workers (int): Number of worker threads used for training.

    Functionality:
        Learns Word2Vec embeddings from the input text data and transforms each document
        into a fixed-length vector by averaging the embeddings of the words it contains.

    Output:
        A fitted vectorizer object in fit(), and a NumPy array of document vectors in transform().
    """
    def __init__(self, vector_size=100, window=5, min_count=5, sg=0, workers=4):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.sg = sg
        self.workers = workers
        self.w2v_model = None

    def fit(self, X, y=None):
        tokenized_texts = [self._tokenize(text) for text in X]
        self.w2v_model = Word2Vec(
            sentences=tokenized_texts,
            vector_size=self.vector_size,
            window=self.window,
            min_count=self.min_count,
            sg=self.sg,
            workers=self.workers
        )
        return self

    def transform(self, X):
        tokenized_texts = [self._tokenize(text) for text in X]
        return np.array([self._document_vector(tokens) for tokens in tokenized_texts])

    def _tokenize(self, text):
        return str(text).lower().split()

    def _document_vector(self, tokens):
        vectors = [self.w2v_model.wv[word] for word in tokens if word in self.w2v_model.wv]
        if len(vectors) == 0:
            return np.zeros(self.vector_size)
        return np.mean(vectors, axis=0)

class TfidfWeightedWord2VecVectorizer(BaseEstimator, TransformerMixin):
    """
    Input:
        vector_size (int): Dimension of word embeddings.
        window (int): Maximum distance between the current and predicted word.
        min_count (int): Minimum frequency required for a word to be included.
        sg (int): Training algorithm, where 0 = CBOW and 1 = Skip-gram.
        workers (int): Number of worker threads used for training.

    Functionality:
        Learns Word2Vec embeddings and TF-IDF weights from the training text data,
        then transforms each document into a fixed-length vector using a TF-IDF-weighted
        average of word embeddings, so that more informative words contribute more.

    Output:
        A fitted vectorizer object in fit(), and a NumPy array of weighted document vectors in transform().
    """
    def __init__(self, vector_size=100, window=5, min_count=2, sg=1, workers=4):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.sg = sg
        self.workers = workers
        self.w2v_model = None
        self.tfidf_vectorizer = None
        self.word2weight = {}

    def _tokenize(self, text):
        text = str(text).lower()
        return re.findall(r"\b\w+\b", text)

    def fit(self, X, y=None):
        tokenized_texts = [self._tokenize(text) for text in X]

        self.w2v_model = Word2Vec(
            sentences=tokenized_texts,
            vector_size=self.vector_size,
            window=self.window,
            min_count=self.min_count,
            sg=self.sg,
            workers=self.workers
        )

        joined_texts = [" ".join(tokens) for tokens in tokenized_texts]
        self.tfidf_vectorizer = TfidfVectorizer(tokenizer=str.split, lowercase=False)
        self.tfidf_vectorizer.fit(joined_texts)

        max_idf = max(self.tfidf_vectorizer.idf_)
        self.word2weight = {
            word: self.tfidf_vectorizer.idf_[idx]
            for word, idx in self.tfidf_vectorizer.vocabulary_.items()
        }
        self.max_idf = max_idf

        return self

    def transform(self, X):
        tokenized_texts = [self._tokenize(text) for text in X]
        return np.array([self._document_vector(tokens) for tokens in tokenized_texts])

    def _document_vector(self, tokens):
        weighted_vectors = []
        weights = []

        for word in tokens:
            if word in self.w2v_model.wv:
                weight = self.word2weight.get(word, self.max_idf)
                weighted_vectors.append(self.w2v_model.wv[word] * weight)
                weights.append(weight)

        if len(weighted_vectors) == 0:
            return np.zeros(self.vector_size)

        return np.sum(weighted_vectors, axis=0) / np.sum(weights)

## Training Method Discriminative Code
This block defines the main training function for the discriminative text classification approach. It loads the training and validation data, trains multiple Logistic Regression pipelines with different text representations, tunes their hyperparameters using grid search, compares their validation performance, selects the best-performing model, and saves the final classifier and vectorizer for later testing.

In [ ]:
def train_dis(train_file, val_file, model_dir):
    """
    Input:
        train_file (str): Path to the training CSV file.
        val_file (str): Path to the validation CSV file.
        model_dir (str): Directory where the final classifier and vectorizer will be saved.

    Functionality:
        Loads the training and validation datasets, handles missing text values,
        trains and tunes multiple Logistic Regression pipelines using different
        text vectorization methods (TF-IDF, Bag of Words, Word2Vec, and TF-IDF
        weighted Word2Vec), evaluates them on the validation set, compares their
        performance, selects the best model based on validation F1-score, and
        saves the final classifier and vectorizer to disk.

    Output:
        None. Saves the selected classifier and vectorizer files in the specified model directory.
    """
    # Read training and validation set data
    train_df = pd.read_csv(train_file)
    val_df = pd.read_csv(val_file)
    print('Training set has', len(train_df),'data points')
    print('Validation set has', len(val_df),'data points')

    # Conver NaN values to empty strings
    train_df['text'] = train_df['text'].fillna('').astype(str)
    val_df['text'] = val_df['text'].fillna('').astype(str)

    # Results dataframe
    results_df = pd.DataFrame(columns=["vectorizer", "valid_accuracy", "valid_F1_score", "grid_search"])

    # TF-IDF Vecotorizer
    tfidf_vectorizer = TfidfVectorizer(lowercase=True)
    tfidf_model = LogisticRegression(max_iter=1000)

    tfidf_pipeline = Pipeline([
        ('vectorizer', tfidf_vectorizer),
        ('classifier', tfidf_model)
    ])

    tfidf_param_grid = {
        'vectorizer__ngram_range': [(1, 2)],
        'vectorizer__min_df': [2, 5, 10],
        'vectorizer__max_df': [0.9],
        'vectorizer__max_features': [5000, 10000],
        'vectorizer__sublinear_tf': [True],
        'classifier__C': [0.001, 0.01, 0.1, 1],
        'classifier__penalty': ['l2'],
        'classifier__solver': ['liblinear'],
        'classifier__class_weight': ['balanced']
    }

    tfidf_grid_search = GridSearchCV(
        estimator=tfidf_pipeline,
        param_grid=tfidf_param_grid,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    )
    # Train model based on TF-IDF representation
    results_df = train_classifier('TFIDF', tfidf_grid_search, train_df, val_df, results_df)

    # Bag of Words
    bow_vectorizer = CountVectorizer(lowercase=True)
    bow_model = LogisticRegression(max_iter=1000)

    bow_pipeline = Pipeline([
        ('vectorizer', bow_vectorizer),
        ('classifier', bow_model)
    ])

    # bow_param_grid = {
    #     # Core feature extraction
    #     'vectorizer__ngram_range': [(1, 1), (1, 2)],
    #     'vectorizer__min_df': [2, 5, 10],
    #     'vectorizer__max_df': [0.9, 1.0],
    #     'vectorizer__max_features': [3000, 10000],
    #     'vectorizer__binary': [False, True],

    #     # Preprocessing options
    #     'vectorizer__lowercase': [True, False],
    #     'vectorizer__stop_words': [None, 'english'],

    #     # Tokenization behavior
    #     'vectorizer__token_pattern': [r'(?u)\b\w\w+\b', r'(?u)\b\w+\b'],  # include short words or not
    #     'vectorizer__strip_accents': [None, 'unicode'],

    #     # Classifier hyperparameters
    #     'classifier__C': [0.001, 0.01, 0.1, 1],
    #     'classifier__penalty': ['l2'],
    #     'classifier__solver': ['liblinear'],
    #     'classifier__class_weight': ['balanced']
    # }

    # This is best param. Use this for faster training
    bow_param_grid = {
        'classifier__C': [0.1],
        'classifier__class_weight': ['balanced'],
        'classifier__penalty': ['l2'],
        'classifier__solver': ['liblinear'],
        'vectorizer__binary': [False],
        'vectorizer__lowercase': [True],
        'vectorizer__max_df': [0.9],
        'vectorizer__max_features': [10000],
        'vectorizer__min_df': [2],
        'vectorizer__ngram_range': [(1, 2)],
        'vectorizer__stop_words': [None],
        'vectorizer__strip_accents': [None],
        'vectorizer__token_pattern': ['(?u)\\b\\w\\w+\\b']}


    bow_grid_search = GridSearchCV(
        estimator=bow_pipeline,
        param_grid=bow_param_grid,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    )
    # Train model based on BoW representation
    results_df = train_classifier('BoW', bow_grid_search, train_df, val_df, results_df)

    # Word2Vec
    w2v_vectorizer = Word2VecVectorizer()
    w2v_model = LogisticRegression(max_iter=1000)

    w2v_pipeline = Pipeline([
        ('vectorizer', w2v_vectorizer),
        ('classifier', w2v_model)
    ])

    w2v_param_grid = {
        'vectorizer__vector_size': [50, 100],
        'vectorizer__window': [3, 5],
        'vectorizer__min_count': [5, 10],
        'vectorizer__sg': [0],
        'classifier__C': [0.001, 0.01, 0.1],
        'classifier__penalty': ['l2'],
        'classifier__solver': ['liblinear'],
        'classifier__class_weight': ['balanced']
    }

    w2v_grid_search = GridSearchCV(
        estimator=w2v_pipeline,
        param_grid=w2v_param_grid,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    )
    # Train model based on Word2vec representation
    results_df = train_classifier('Word2Vec', w2v_grid_search, train_df, val_df, results_df)

    # TF-IDF Weighted Word2Vec
    tfidf_w2v_vectorizer = TfidfWeightedWord2VecVectorizer()
    tfidf_w2v_model = LogisticRegression(max_iter=2000)

    tfidf_w2v_pipeline = Pipeline([
        ('vectorizer', tfidf_w2v_vectorizer),
        ('classifier', tfidf_w2v_model)
    ])

    tfidf_w2v_param_grid = {
        'vectorizer__vector_size': [100, 200],
        'vectorizer__window': [3, 5],
        'vectorizer__min_count': [2, 3],
        'vectorizer__sg': [0, 1],
        'classifier__C': [0.01, 0.1, 1],
        'classifier__penalty': ['l2'],
        'classifier__solver': ['liblinear'],
        'classifier__class_weight': ['balanced']
    }

    # This is best param. Use this for faster training
    # tfidf_w2v_param_grid = {
    #     'vectorizer__vector_size': [100],
    #     'vectorizer__window': [5],
    #     'vectorizer__min_count': [3],
    #     'vectorizer__sg': [1],
    #     'classifier__C': [1],
    #     'classifier__penalty': ['l2'],
    #     'classifier__solver': ['liblinear'],
    #     'classifier__class_weight': ['balanced']
    # }


    tfidf_w2v_grid_search = GridSearchCV(
        estimator=tfidf_w2v_pipeline,
        param_grid=tfidf_w2v_param_grid,
        cv=5,
        scoring='f1_macro',
        n_jobs=1
    )
    # Train model based on TF-IDF Weighted Word2vec representation
    results_df = train_classifier('TFIDFWeightedWord2Vec', tfidf_w2v_grid_search, train_df, val_df, results_df)

    # Model selection using validation set
    results_df = results_df.sort_values(by='valid_F1_score', ascending=False)
    best_model = results_df.iloc[0]

    # Print model summary
    print('----------------- Model Performance Summary -----------------')
    print(results_df[['vectorizer', 'valid_accuracy', 'valid_F1_score']])

    # Select the best model
    print('\n------------------------ Final Model ------------------------')
    print(f"Best model based on validation F1-score is {best_model['vectorizer']} with F1-score of {round(best_model['valid_F1_score'] * 100):.0f}%")

    # Save classifier
    classifier = best_model['grid_search'].best_estimator_.named_steps['classifier']
    model_file = os.path.join(model_dir, 'model.sav')
    pickle.dump(classifier, open(model_file, 'wb'))
    print('Saved model to', model_file)

    # Save vectorizer
    vectorizer = best_model['grid_search'].best_estimator_.named_steps['vectorizer']
    vocab_file = os.path.join(model_dir, 'vocab.sav')
    pickle.dump(vectorizer, open(vocab_file, 'wb'))
    print('Saved vocab to', vocab_file)

    return

In [ ]:
train_dis(train_file, val_file, MODEL_Dis_DIRECTORY)

Training set has 4914 data points
Validation set has 700 data points
Start Training  TFIDF
Best parameters:
 {'classifier__C': 1, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear', 'vectorizer__max_df': 0.9, 'vectorizer__max_features': 10000, 'vectorizer__min_df': 2, 'vectorizer__ngram_range': (1, 2), 'vectorizer__sublinear_tf': True}
Train F1-macro: 0.89 %
Validation F1-macro: 0.82 %
Finish Training  TFIDF
Start Training  BoW
Best parameters:
 {'classifier__C': 0.1, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear', 'vectorizer__binary': False, 'vectorizer__lowercase': True, 'vectorizer__max_df': 0.9, 'vectorizer__max_features': 10000, 'vectorizer__min_df': 2, 'vectorizer__ngram_range': (1, 2), 'vectorizer__stop_words': None, 'vectorizer__strip_accents': None, 'vectorizer__token_pattern': '(?u)\\b\\w\\w+\\b'}
Train F1-macro: 0.93 %
Validation F1-macro: 0.84 %
Finish Training  BoW


## Testing Method Discriminative Code
This block defines the testing function for the discriminative approach. It loads the saved classifier and vectorizer, transforms the test data into feature vectors, generates predictions, saves them to the test file, and optionally evaluates the model using F1-score if ground-truth labels are available.

In [ ]:
def test_dis(test_file, model_dir):
    """
    Input:
        test_file (str): Path to the test CSV file containing the input text data.
        model_dir (str): Directory where the trained classifier and vectorizer are stored.

    Functionality:
        Loads the test dataset, handles missing text values, loads the saved
        Logistic Regression classifier and vectorizer, transforms the text data
        into feature vectors, generates predictions, saves the predictions to
        the test file, and computes the macro F1-score if the sentiment column exists.

    Output:
        None. Saves the updated test file with predictions and optionally prints evaluation metrics.
    """

    # Read test data
    test_df = read_data(test_file)
    print('Testing set has', len(test_df),'data points')
    test_df['text'] = test_df['text'].fillna('').astype(str)

    # Load classifier
    model_file = model_dir +'/model.sav'
    classifier = pickle.load(open(model_file, 'rb'))
    print('Model loaded from', model_file)

    # Load vectorizer
    vocab_file = model_dir +'/vocab.sav'
    count_vectorizer = pickle.load(open(vocab_file, 'rb'))
    print('Vocab loaded from ', vocab_file)

    # Predicting on the test data
    test_counts = count_vectorizer.transform(test_df['text'])
    predictions = classifier.predict(test_counts)
    print('Testing Done')

    # Save predictions
    test_df['out_label_LR'] = predictions
    test_df.to_csv(test_file,index=False)
    print('Saved output to ',test_file)

    # Calculate F1-score for test data
    try:
        target = test_df['sentiment']
        score=f1_score(target, predictions, average='macro')
        print('macro F1 Score on Test set', score)
    except KeyError:
        print('sentiment column is not provided in test file to calculate F1 Score.')

    return test_df


In [ ]:
# RUN FOR PRESENTATION
df = test_dis(test_file, MODEL_Dis_DIRECTORY)
df.head(5)

gdrive/MyDrive/./CE807-26-SP/Assignment/data/6/test.csv has 1383 data points
Testing set has 1383 data points
Model loaded from gdrive/MyDrive/./CE807-26-SP/Assignment/model/2501676/model_dis/model.sav
Vocab loaded from  gdrive/MyDrive/./CE807-26-SP/Assignment/model/2501676/model_dis/vocab.sav
Testing Done
Saved output to  gdrive/MyDrive/./CE807-26-SP/Assignment/data/6/test.csv
sentiment column is not provided in test file to calculate F1 Score.


,text,data_id,out_label_Prompt_1,out_label_Prompt_2,out_label_Prompt_3,out_label_Prompt_4,out_label_Prompt_5,out_label_LR
0,Everything was as expected.<br />Delivery and ...,6,positive,positive,positive,positive,positive,positive
1,works,6,positive,positive,positive,positive,positive,positive
2,Easy Replacement Parts for Appliances at a Red...,6,positive,positive,positive,positive,positive,positive
3,Love this but be careful not to over fill with...,6,positive,positive,positive,positive,positive,negative
4,fits perfectly,6,positive,positive,positive,positive,positive,positive


## Discriminative Method  End


# Performance analysis by text length across models

This block defines an analysis function that evaluates model performance across different text length groups. It bins the data based on word count, computes weighted F1-scores for each model within each bin, and compares how Logistic Regression and prompt-based models perform on texts of varying lengths.

In [ ]:
# For running this function you should run test_unsup() and test_dis() for val_file to save predictions in valid.csv. Then run this. (The final result is already exist in report)
def analysis(file):
    """
    Input:
        val_file (str): Path to the dataset file containing text, sentiment, and model prediction columns.

    Functionality:
        Loads the dataset, computes the word length of each text, groups the data
        into predefined length bins, and calculates the weighted F1-score for each
        model within each bin to analyze performance variation across text lengths.

    Output:
        None. Prints a dataframe showing F1-scores per model for each length bin.
    """
    # Read data
    df = read_data(file)
    df["word_len"] = df["text"].astype(str).str.split().str.len()

    bins = [0, 8, 15, 30, 45, float("inf")]
    labels = ["1-8", "9-15", "16-30", "31-45", "46+"]
    df["len_bin"] = pd.cut(df["word_len"], bins=bins, labels=labels, include_lowest=True)

    models = [
        "out_label_LR",
        "out_label_Prompt_1",
        "out_label_Prompt_2",
        "out_label_Prompt_3",
        "out_label_Prompt_4",
        "out_label_Prompt_5"
    ]

    result_df = pd.DataFrame({
        model: df.groupby("len_bin").apply(
            lambda x: f1_score(x["sentiment"], x[model], average="weighted")
        )
        for model in models
    })

    print(result_df)